In [ ]:
from pathlib import Path
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from allensdk.brain_observatory.behavior.behavior_project_cache.\
    behavior_neuropixels_project_cache import (
        VisualBehaviorNeuropixelsProjectCache
    )

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

ALLEN_CACHE_DIR = Path("allen_visual_behavior_neuropixels_cache")
ALLEN_CACHE_DIR.mkdir(parents=True, exist_ok=True)

Path("outputs/tables").mkdir(parents=True, exist_ok=True)
Path("outputs/figures").mkdir(parents=True, exist_ok=True)

print("Cache directory:", ALLEN_CACHE_DIR.resolve())

In [ ]:
cache = VisualBehaviorNeuropixelsProjectCache.from_local_cache(
    cache_dir=str(ALLEN_CACHE_DIR),
    use_static_cache=True
)

ecephys_sessions = cache.get_ecephys_session_table()

print("Number of available ecephys sessions:", len(ecephys_sessions))
print("\nColumns:")
print(ecephys_sessions.columns.tolist())

display(ecephys_sessions.head())

In [ ]:
session_summary_columns = [
    column for column in [
        "behavior_session_id",
        "date_of_acquisition",
        "mouse_id",
        "sex",
        "genotype",
        "session_type",
        "experience_level",
        "image_set",
        "unit_count",
        "channel_count",
        "structure_acronyms",
        "abnormal_activity",
        "abnormal_histology"
    ]
    if column in ecephys_sessions.columns
]

display(
    ecephys_sessions[
        session_summary_columns
    ].head(10)
)

In [ ]:
if "experience_level" in ecephys_sessions.columns:
    print(
        ecephys_sessions["experience_level"]
        .value_counts(dropna=False)
    )

print("\nUnit-count summary:")
print(ecephys_sessions["unit_count"].describe())

In [ ]:
candidate_sessions = ecephys_sessions.copy()

if "abnormal_activity" in candidate_sessions.columns:
    candidate_sessions = candidate_sessions[
        candidate_sessions["abnormal_activity"].isna()
    ]

if "abnormal_histology" in candidate_sessions.columns:
    candidate_sessions = candidate_sessions[
        candidate_sessions["abnormal_histology"].isna()
    ]

candidate_sessions = candidate_sessions[
    candidate_sessions["unit_count"] >= 500
]

if "experience_level" in candidate_sessions.columns:
    familiar_candidates = candidate_sessions[
        candidate_sessions["experience_level"] == "Familiar"
    ]

    if len(familiar_candidates) > 0:
        candidate_sessions = familiar_candidates.copy()

candidate_sessions = candidate_sessions[
    candidate_sessions["structure_acronyms"].apply(
        lambda structures: "VISp" in structures
        if isinstance(structures, (list, np.ndarray))
        else False
    )
]

candidate_sessions = candidate_sessions.sort_values(
    "unit_count",
    ascending=False
)

print("Eligible candidate sessions:", len(candidate_sessions))

selection_columns = [
    column for column in [
        "mouse_id",
        "sex",
        "genotype",
        "experience_level",
        "image_set",
        "unit_count",
        "structure_acronyms"
    ]
    if column in candidate_sessions.columns
]

display(candidate_sessions[selection_columns].head(10))

In [ ]:
SELECTED_SESSION_ID = int(candidate_sessions.index[0])

print("Selected ecephys session ID:", SELECTED_SESSION_ID)
display(
    candidate_sessions.loc[
        [SELECTED_SESSION_ID],
        selection_columns
    ]
)

In [ ]:
session = cache.get_ecephys_session(
    ecephys_session_id=SELECTED_SESSION_ID
)

print("Session loaded.")
print("\nSession metadata:")
print(session.metadata)

In [ ]:
trials = session.trials.copy()
stimulus_presentations = session.stimulus_presentations.copy()
licks = session.licks.copy()
rewards = session.rewards.copy()

print("Trials shape:", trials.shape)
print("Stimulus presentations shape:", stimulus_presentations.shape)
print("Licks shape:", licks.shape)
print("Rewards shape:", rewards.shape)

print("\nTrials columns:")
print(trials.columns.tolist())

print("\nStimulus-presentations columns:")
print(stimulus_presentations.columns.tolist())

In [ ]:
import h5py
import numpy as np
import pandas as pd
from pathlib import Path

OUTPUT_TABLE_DIR = Path("outputs/tables")
OUTPUT_TABLE_DIR.mkdir(parents=True, exist_ok=True)

def decode_text(values):
    return np.array([
        value.decode("utf-8") if isinstance(value, bytes) else value
        for value in values
    ])

with h5py.File(NWB_PATH, "r") as h5_file:
    trial_group = h5_file["intervals"]["trials"]

    mouse_trials = pd.DataFrame({
        "trial_id": trial_group["id"][:],
        "start_time": trial_group["start_time"][:],
        "stop_time": trial_group["stop_time"][:],
        "trial_length": trial_group["trial_length"][:],
        "initial_image": decode_text(
            trial_group["initial_image_name"][:]
        ),
        "change_image": decode_text(
            trial_group["change_image_name"][:]
        ),
        "is_change": trial_group["is_change"][:].astype(int),
        "is_go": trial_group["go"][:].astype(int),
        "is_catch": trial_group["catch"][:].astype(int),
        "change_time": trial_group[
            "change_time_no_display_delay"
        ][:],
        "response_time": trial_group["response_time"][:],
        "reward_time": trial_group["reward_time"][:],
        "reward_volume": trial_group["reward_volume"][:],
        "hit": trial_group["hit"][:].astype(int),
        "miss": trial_group["miss"][:].astype(int),
        "false_alarm": trial_group["false_alarm"][:].astype(int),
        "correct_rejection": trial_group[
            "correct_reject"
        ][:].astype(int),
        "aborted": trial_group["aborted"][:].astype(int),
        "auto_rewarded": trial_group["auto_rewarded"][:].astype(int)
    })

mouse_trials["response_lick"] = (
    mouse_trials["hit"] + mouse_trials["false_alarm"]
).clip(upper=1)

mouse_trials["correct"] = (
    mouse_trials["hit"] + mouse_trials["correct_rejection"]
).clip(upper=1)

mouse_trials["outcome"] = np.select(
    [
        mouse_trials["hit"].eq(1),
        mouse_trials["miss"].eq(1),
        mouse_trials["false_alarm"].eq(1),
        mouse_trials["correct_rejection"].eq(1),
        mouse_trials["aborted"].eq(1)
    ],
    [
        "hit",
        "miss",
        "false_alarm",
        "correct_rejection",
        "aborted"
    ],
    default="other"
)

mouse_trials["reaction_time"] = (
    mouse_trials["response_time"] -
    mouse_trials["change_time"]
)

mouse_trials.to_csv(
    OUTPUT_TABLE_DIR / "allen_mouse_trials_raw.csv",
    index=False
)

print("Trial-table shape:", mouse_trials.shape)
print("\nOutcome counts:")
print(mouse_trials["outcome"].value_counts(dropna=False))

display(mouse_trials.head())

In [ ]:
valid_trials = mouse_trials[
    (mouse_trials["aborted"] == 0)
    & (mouse_trials["auto_rewarded"] == 0)
].copy()

print("Valid trials:", len(valid_trials))
print("Excluded trials:", len(mouse_trials) - len(valid_trials))

print("\nOutcome counts, valid trials:")
print(valid_trials["outcome"].value_counts())

print("\nCheck outcome labels sum to valid trials:")
print(
    valid_trials[
        ["hit", "miss", "false_alarm", "correct_rejection"]
    ].sum(axis=1).value_counts(dropna=False)
)

print("\nReaction-time summary, lick-response trials:")
print(
    valid_trials.loc[
        valid_trials["response_lick"] == 1,
        "reaction_time"
    ].describe()
)

In [ ]:
from scipy.stats import norm

behavior_counts = (
    valid_trials["outcome"]
    .value_counts()
    .reindex(
        ["hit", "miss", "false_alarm", "correct_rejection"],
        fill_value=0
    )
)

n_go = int(
    (valid_trials["outcome"].isin(["hit", "miss"])).sum()
)

n_catch = int(
    valid_trials["outcome"]
    .isin(["false_alarm", "correct_rejection"])
    .sum()
)

n_hits = int(behavior_counts["hit"])
n_false_alarms = int(behavior_counts["false_alarm"])

# Log-linear correction prevents infinite z-scores if a rate is 0 or 1
hit_rate_corrected = (n_hits + 0.5) / (n_go + 1)
false_alarm_rate_corrected = (
    (n_false_alarms + 0.5) / (n_catch + 1)
)

mouse_d_prime = (
    norm.ppf(hit_rate_corrected) -
    norm.ppf(false_alarm_rate_corrected)
)

mouse_criterion = -0.5 * (
    norm.ppf(hit_rate_corrected) +
    norm.ppf(false_alarm_rate_corrected)
)

mouse_behavior_summary = pd.DataFrame([{
    "session_id": 1119946360,
    "n_valid_trials": len(valid_trials),
    "n_go_trials": n_go,
    "n_catch_trials": n_catch,
    "hit_rate_raw": n_hits / n_go,
    "false_alarm_rate_raw": n_false_alarms / n_catch,
    "hit_rate_corrected": hit_rate_corrected,
    "false_alarm_rate_corrected": false_alarm_rate_corrected,
    "d_prime": mouse_d_prime,
    "criterion": mouse_criterion,
    "median_reaction_time_s": valid_trials.loc[
        valid_trials["response_lick"] == 1,
        "reaction_time"
    ].median()
}])

display(mouse_behavior_summary)

mouse_behavior_summary.to_csv(
    OUTPUT_TABLE_DIR / "allen_mouse_behavior_summary.csv",
    index=False
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

OUTPUT_FIGURE_DIR = Path("outputs/figures")
OUTPUT_FIGURE_DIR.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 4.5)
)

outcome_order = [
    "hit",
    "miss",
    "false_alarm",
    "correct_rejection"
]

sns.countplot(
    data=valid_trials,
    x="outcome",
    order=outcome_order,
    ax=axes[0],
    palette={
        "hit": "#1f77b4",
        "miss": "#d62728",
        "false_alarm": "#ff7f0e",
        "correct_rejection": "#2ca02c"
    }
)

axes[0].set_title("Mouse change-detection outcomes")
axes[0].set_xlabel("")
axes[0].set_ylabel("Number of valid trials")
axes[0].tick_params(axis="x", rotation=20)

reaction_times = valid_trials.loc[
    valid_trials["response_lick"] == 1,
    "reaction_time"
].dropna()

sns.histplot(
    reaction_times,
    bins=20,
    kde=True,
    ax=axes[1],
    color="#1f77b4"
)

axes[1].axvline(
    reaction_times.median(),
    color="black",
    linestyle="--",
    label=f"Median = {reaction_times.median():.3f} s"
)

axes[1].set_title("Mouse lick reaction times")
axes[1].set_xlabel("Reaction time after change (s)")
axes[1].set_ylabel("Trial count")
axes[1].legend()

plt.tight_layout()

plt.savefig(
    OUTPUT_FIGURE_DIR / "allen_mouse_behavior_summary.png",
    dpi=250,
    bbox_inches="tight"
)

plt.show()